# 🧬 SpectralBio V2 — Maximum Performance Pipeline

**Claw4S 2026** | Stanford & Princeton

**Authors:** Claw 🦞 (AI Co-author), Davi Bonetto

---

### Improvements over V1:
1. **Proper Log-Likelihood** via `EsmForMaskedLM` (not crude norm proxy)
2. **Layer Selection** — deep layers only (last 10, last 5)
3. **Top-k Eigenvalues** — focus on dominant spectral modes
4. **Log Eigenvalue Ratios** — scale-invariant comparison
5. **Trace Ratio** — overall variance change metric
6. **Frobenius Norm** — matrix-level covariance distance
7. **Optimal Combination** — grid search for best α weights
8. **Optimal F1 Threshold** — not fixed at 0.5
9. **Bootstrap Confidence Intervals** — statistical rigor
10. **Feature Ablation** — which components matter most

**Instructions:** Runtime → Run All → Wait ~20 min → Download results zip

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch transformers scipy scikit-learn pandas numpy matplotlib requests
print('✓ All dependencies installed')

## Step 2: Download and Prepare ClinVar Data

In [ ]:
import urllib.request, gzip, csv, json, os, re, time

os.makedirs('/content/spectralbio_data', exist_ok=True)

print('Downloading ClinVar variant_summary.txt.gz (~80MB)...')
url = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
dest = '/content/spectralbio_data/variant_summary.txt.gz'
if not os.path.exists(dest):
    urllib.request.urlretrieve(url, dest)
    print(f'✓ Downloaded: {dest}')
else:
    print(f'✓ Already exists: {dest}')

THREE_TO_ONE = {
    'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Glu':'E','Gln':'Q',
    'Gly':'G','His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F',
    'Pro':'P','Ser':'S','Thr':'T','Trp':'W','Tyr':'Y','Val':'V'
}

def parse_aa_change(name_field):
    m = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', str(name_field))
    if not m: return None
    wt = THREE_TO_ONE.get(m.group(1))
    pos = int(m.group(2)) - 1
    mut = THREE_TO_ONE.get(m.group(3))
    if wt and mut and wt != mut: return wt, pos, mut
    return None

LABEL_MAP = {'Pathogenic': 1, 'Likely pathogenic': 1, 'Benign': 0, 'Likely benign': 0}

variants = []
with gzip.open(dest, 'rt', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        if row.get('GeneSymbol','').strip() != 'TP53': continue
        sig = row.get('ClinicalSignificance','').strip()
        label = LABEL_MAP.get(sig)
        if label is None: continue
        if 'single nucleotide' not in row.get('Type','').lower(): continue
        parsed = parse_aa_change(row.get('Name',''))
        if parsed is None: continue
        wt_aa, pos, mut_aa = parsed
        variants.append({'gene':'TP53','wt_aa':wt_aa,'position':pos,'mut_aa':mut_aa,'label':label,'name':row.get('Name','')[:80]})

seen = set()
unique_variants = []
for v in variants:
    key = (v['position'], v['mut_aa'])
    if key not in seen:
        seen.add(key)
        unique_variants.append(v)

with open('/content/spectralbio_data/tp53_variants.json','w') as f:
    json.dump(unique_variants, f, indent=2)
n_path = sum(1 for v in unique_variants if v['label']==1)
n_ben = sum(1 for v in unique_variants if v['label']==0)
print(f'✓ TP53 variants: {len(unique_variants)} total ({n_path} pathogenic, {n_ben} benign)')

## Step 3: Download TP53 Wild-Type Sequence

In [ ]:
print('Fetching TP53 sequence from UniProt (P04637)...')
with urllib.request.urlopen('https://www.uniprot.org/uniprot/P04637.fasta') as r:
    fasta = r.read().decode('utf-8')
sequence = ''.join(fasta.strip().split('\n')[1:])
assert len(sequence) == 393, f'Expected 393 aa, got {len(sequence)}'
print(f'✓ TP53: {len(sequence)} aa')
with open('/content/spectralbio_data/tp53_sequence.txt','w') as f: f.write(sequence)

## Step 4: Load ESM2-150M (with Masked LM Head)

**V2 improvement:** Using `EsmForMaskedLM` instead of `EsmModel`.
This gives us BOTH hidden states AND proper log-likelihood predictions.

In [ ]:
import torch
import numpy as np
import random
from transformers import EsmForMaskedLM, EsmTokenizer

torch.manual_seed(42); np.random.seed(42); random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}, {props.total_memory / 1e9:.1f}GB')

MODEL_NAME = 'facebook/esm2_t30_150M_UR50D'
print(f'Loading {MODEL_NAME} with MaskedLM head...')
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
model = EsmForMaskedLM.from_pretrained(MODEL_NAME, output_hidden_states=True)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_layers = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
print(f'✓ Model loaded: {n_params:.0f}M params, {n_layers} layers, dim={hidden_dim}')
print(f'✓ Has LM head for proper log-likelihood computation')

## Step 5: Compute ALL Features Per Variant

For each variant we compute **11 features** in a single forward pass:

| # | Feature | Description |
|---|---------|-------------|
| 1 | `sps_all` | Original SPS (all 30 layers) |
| 2 | `sps_deep10` | SPS using only last 10 layers |
| 3 | `sps_deep5` | SPS using only last 5 layers |
| 4 | `sps_topk` | SPS using top-20 eigenvalues only |
| 5 | `sps_log` | Log-space eigenvalue comparison |
| 6 | `trace_ratio` | Mean tr(C_mut)/tr(C_wt) across layers |
| 7 | `frob_dist` | Mean Frobenius norm of cov difference |
| 8 | `ll_crude` | Norm-based LL proxy (V1 baseline) |
| 9 | `ll_proper` | Actual masked LM log-likelihood ratio |
| 10 | `pos_entropy` | Prediction entropy at mutation site |
| 11 | `mut_rank` | Rank of mutant aa among 20 possible |

⏱ **~15-20 min on T4 GPU, ~30 min on CPU**

In [ ]:
from scipy.linalg import eigvalsh

torch.manual_seed(42); np.random.seed(42); random.seed(42)

with open('/content/spectralbio_data/tp53_variants.json') as f:
    variants_loaded = json.load(f)
with open('/content/spectralbio_data/tp53_sequence.txt') as f:
    wt_seq = f.read().strip()

WINDOW = 40
TOPK = 20
AA_TOKENS = list('ACDEFGHIKLMNPQRSTVWY')
AA_TOKEN_IDS = {aa: tokenizer.convert_tokens_to_ids(aa) for aa in AA_TOKENS}

def forward_pass(seq, center_pos):
    """Single forward pass returning hidden states + logits for local window."""
    start = max(0, center_pos - WINDOW)
    end = min(len(seq), center_pos + WINDOW)
    local_seq = seq[start:end]
    local_mut_pos = center_pos - start
    inputs = tokenizer(local_seq, return_tensors='pt', add_special_tokens=True, padding=False).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    hidden = torch.stack(outputs.hidden_states[1:], dim=0)[:, 0, 1:-1, :].cpu().numpy()
    logits = outputs.logits[0].cpu()
    return hidden, logits, local_mut_pos

def compute_spectral_features(hidden_wt, hidden_mut, logits_wt, local_pos, wt_aa, mut_aa):
    """Compute all 11 features from a WT/MUT pair."""
    n_layers = hidden_wt.shape[0]
    shifts_all = []
    shifts_topk = []
    shifts_log = []
    trace_ratios = []
    frob_dists = []

    for l in range(n_layers):
        cov_wt = np.cov(hidden_wt[l].T)
        cov_mut = np.cov(hidden_mut[l].T)
        ev_wt = np.sort(eigvalsh(cov_wt))
        ev_mut = np.sort(eigvalsh(cov_mut))
        min_len = min(len(ev_wt), len(ev_mut))
        ev_wt_t = ev_wt[:min_len]
        ev_mut_t = ev_mut[:min_len]

        # 1. Full SPS (all eigenvalues)
        shifts_all.append(float(np.sum((ev_mut_t - ev_wt_t)**2)))

        # 2. Top-k SPS
        k = min(TOPK, min_len)
        shifts_topk.append(float(np.sum((ev_mut_t[-k:] - ev_wt_t[-k:])**2)))

        # 3. Log SPS (scale-invariant)
        eps = 1e-10
        log_wt = np.log(np.abs(ev_wt_t) + eps)
        log_mut = np.log(np.abs(ev_mut_t) + eps)
        shifts_log.append(float(np.sum((log_mut - log_wt)**2)))

        # 4. Trace ratio
        tr_wt = np.trace(cov_wt)
        tr_mut = np.trace(cov_mut)
        if abs(tr_wt) > eps:
            trace_ratios.append(abs(tr_mut / tr_wt - 1.0))

        # 5. Frobenius norm
        frob_dists.append(float(np.linalg.norm(cov_mut - cov_wt, 'fro')))

    shifts_all = np.array(shifts_all)

    # SPS variants by layer selection
    sps_all = float(np.mean(shifts_all))
    sps_deep10 = float(np.mean(shifts_all[-10:])) if n_layers >= 10 else sps_all
    sps_deep5 = float(np.mean(shifts_all[-5:])) if n_layers >= 5 else sps_all
    sps_topk = float(np.mean(shifts_topk))
    sps_log = float(np.mean(shifts_log))
    trace_ratio = float(np.mean(trace_ratios)) if trace_ratios else 0.0
    frob_dist = float(np.mean(frob_dists))

    # Crude LL (V1 style - norm difference)
    ll_crude = abs(float(np.linalg.norm(hidden_mut[-1]) - np.linalg.norm(hidden_wt[-1])))

    # Proper LL from masked LM head
    token_pos = local_pos + 1  # +1 for CLS
    log_probs = torch.log_softmax(logits_wt[token_pos], dim=-1)
    wt_id = AA_TOKEN_IDS.get(wt_aa)
    mut_id = AA_TOKEN_IDS.get(mut_aa)
    if wt_id is not None and mut_id is not None:
        ll_proper = float(log_probs[wt_id].item() - log_probs[mut_id].item())
    else:
        ll_proper = 0.0

    # Position entropy
    probs = torch.softmax(logits_wt[token_pos], dim=-1)
    aa_probs = torch.tensor([probs[AA_TOKEN_IDS[aa]].item() for aa in AA_TOKENS])
    aa_probs = aa_probs / aa_probs.sum()
    pos_entropy = float(-torch.sum(aa_probs * torch.log(aa_probs + 1e-10)).item())

    # Mutant rank (1 = most likely, 20 = least likely)
    aa_scores = [(aa, probs[AA_TOKEN_IDS[aa]].item()) for aa in AA_TOKENS]
    aa_scores.sort(key=lambda x: x[1], reverse=True)
    mut_rank = 20
    for rank, (aa, _) in enumerate(aa_scores, 1):
        if aa == mut_aa:
            mut_rank = rank
            break

    return {
        'sps_all': sps_all, 'sps_deep10': sps_deep10, 'sps_deep5': sps_deep5,
        'sps_topk': sps_topk, 'sps_log': sps_log,
        'trace_ratio': trace_ratio, 'frob_dist': frob_dist,
        'll_crude': ll_crude, 'll_proper': ll_proper,
        'pos_entropy': pos_entropy, 'mut_rank': float(mut_rank)
    }

# === MAIN SCORING LOOP ===
print(f'Scoring {len(variants_loaded)} TP53 variants with 11 features each...')
t0 = time.time()
results = []
wt_cache = {}

for i, v in enumerate(variants_loaded):
    pos = v['position']
    if pos >= len(wt_seq) or wt_seq[pos] != v['wt_aa']:
        continue
    mut_seq_full = wt_seq[:pos] + v['mut_aa'] + wt_seq[pos+1:]

    cache_key = (max(0, pos - WINDOW), min(len(wt_seq), pos + WINDOW))
    if cache_key not in wt_cache:
        wt_cache[cache_key] = forward_pass(wt_seq, pos)
    hidden_wt, logits_wt, local_pos = wt_cache[cache_key]
    hidden_mut, _, _ = forward_pass(mut_seq_full, pos)

    features = compute_spectral_features(
        hidden_wt, hidden_mut, logits_wt, local_pos, v['wt_aa'], v['mut_aa']
    )
    features.update({'name': v['name'], 'position': pos,
                     'wt_aa': v['wt_aa'], 'mut_aa': v['mut_aa'], 'label': v['label']})
    results.append(features)

    if (i+1) % 25 == 0 or (i+1) == len(variants_loaded):
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(variants_loaded) - i - 1)
        print(f'  [{i+1}/{len(variants_loaded)}] '
              f"{'PATH' if v['label'] else 'BEN '} "
              f"SPS={features['sps_all']:.2f} "
              f"SPS_d10={features['sps_deep10']:.2f} "
              f"LL_prop={features['ll_proper']:.3f} "
              f'({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)')

with open('/content/spectralbio_data/scores_v2.json', 'w') as f:
    json.dump(results, f, indent=2)

total_time = time.time() - t0
print(f'\n✓ Scored {len(results)} variants × 11 features in {total_time:.0f}s')
print(f'  Pathogenic: {sum(1 for r in results if r["label"]==1)}')
print(f'  Benign:     {sum(1 for r in results if r["label"]==0)}')

## Step 6: Feature Analysis & Ablation Study

Which features are most discriminative? Individual AUC-ROC for each.